[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/19_cadenas_de_markov/notebooks/02_ergodicidad.ipynb)

# Teorema Ergódico en Acción

**Módulo 19 — Cadenas de Markov · Notebook 02**

In [ ]:
# Solo en Colab — en entorno local estas librerías ya deben estar instaladas
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue":   "#2E86AB",
    "red":    "#E94F37",
    "green":  "#27AE60",
    "gray":   "#7F8C8D",
    "orange": "#F39C12",
    "purple": "#8E44AD",
    "dark":   "#2C3E50",
    "teal":   "#1ABC9C",
}

np.random.seed(42)

In [ ]:
# =========================================================================
# Clase CadenaDeMarkov  (auto-contenida: se reproduce aquí para que el
# notebook funcione de forma independiente)
# =========================================================================

class CadenaDeMarkov:
    """Cadena de Markov de tiempo discreto con espacio de estados finito."""

    def __init__(self, P, labels=None, colors=None):
        self.P = np.array(P, dtype=float)
        self.k = self.P.shape[0]
        self.labels = labels or [str(i) for i in range(self.k)]
        self.colors = colors or [f"C{i}" for i in range(self.k)]

    # --- simulación ---
    def simular(self, start, n_steps, rng=None):
        """Devuelve array de estados de longitud n_steps + 1."""
        if rng is None:
            rng = np.random.default_rng(42)
        states = [start]
        for _ in range(n_steps):
            states.append(rng.choice(self.k, p=self.P[states[-1]]))
        return np.array(states)

    # --- distribución estacionaria ---
    def estacionaria(self):
        """pi tal que pi P = pi, normalizada."""
        eigenvalues, eigenvectors = np.linalg.eig(self.P.T)
        idx = np.argmin(np.abs(eigenvalues - 1.0))
        pi = np.real(eigenvectors[:, idx])
        return pi / pi.sum()


# =========================================================================
# Matrices de transición
# =========================================================================

# Vocales / Consonantes  (análisis de texto en español, Markov 1913)
P_VC = np.array([
    [0.35, 0.65],   # V -> V, V -> C
    [0.52, 0.48],   # C -> V, C -> C
])
VC_LABELS = ["V", "C"]
VC_COLORS = [COLORS["blue"], COLORS["red"]]

# Regímenes de mercado
# P_MARKET: matriz canónica del módulo (misma que lab_markov.py).
# En el Notebook 01 se usa una matriz distinta (P_gen) para generar
# datos sintéticos y luego estimar P desde los datos — aquí usamos
# P_MARKET directamente para las demostraciones teóricas.
P_MARKET = np.array([
    [0.70, 0.15, 0.15],
    [0.10, 0.65, 0.25],
    [0.20, 0.15, 0.65],
])
MARKET_LABELS = ["Alcista", "Bajista", "Lateral"]
MARKET_COLORS = [COLORS["green"], COLORS["red"], COLORS["orange"]]

cadena_vc = CadenaDeMarkov(P_VC, VC_LABELS, VC_COLORS)
pi_vc = cadena_vc.estacionaria()
print(f"Distribución estacionaria V/C: pi_V = {pi_vc[0]:.4f},  pi_C = {pi_vc[1]:.4f}")

## Parte 1 — Convergencia del promedio temporal desde múltiples inicios

El **teorema ergódico** dice: sin importar dónde empieces, la fracción de tiempo
que la cadena pasa en cada estado converge a $\pi$.  Vamos a verlo.

In [ ]:
n_steps = 2000
n_chains = 12

fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.cm.tab10

for i in range(n_chains):
    rng = np.random.default_rng(i * 7 + 1)
    start = i % 2          # alternar V y C
    chain = cadena_vc.simular(start, n_steps, rng=rng)
    freq_v = np.cumsum(chain == 0) / np.arange(1, len(chain) + 1)
    ax.plot(freq_v, color=cmap(i / n_chains), alpha=0.6, linewidth=1)

ax.axhline(pi_vc[0], color=COLORS["dark"], linestyle="--", linewidth=2.5,
           label=f"$\\pi_V = {pi_vc[0]:.3f}$", zorder=10)

ax.set_xlabel("Paso ($t$)", fontsize=12)
ax.set_ylabel("Fracción de tiempo en V", fontsize=12)
ax.set_title("Teorema ergódico: 12 cadenas desde distintos inicios \u2192 "
             "todas convergen a $\\pi_V$",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, 1)
ax.legend(fontsize=12, loc="upper right")
plt.tight_layout()
plt.show()

## Parte 2 — Acoplamiento visual

**Acoplamiento (*coupling*)**: dos cadenas con la **misma** $P$ y los **mismos** números
aleatorios, pero **distintos** estados iniciales.  Observa cómo se fusionan.

In [ ]:
rng = np.random.default_rng(123)
n_steps = 40

# Números aleatorios compartidos
us = rng.random(n_steps)

chain_x = [0]   # comienza en V
chain_y = [1]   # comienza en C
coupling_time = None

for t in range(n_steps):
    u = us[t]
    # Transición para X
    next_x = 0 if u < P_VC[chain_x[-1], 0] else 1
    # Transición para Y (mismo u)
    next_y = 0 if u < P_VC[chain_y[-1], 0] else 1

    chain_x.append(next_x)
    chain_y.append(next_y)

    if coupling_time is None and next_x == next_y:
        coupling_time = t + 1

chain_x = np.array(chain_x)
chain_y = np.array(chain_y)

# --- Gráfica: dos cintas horizontales ---
fig, axes = plt.subplots(2, 1, figsize=(14, 4), sharex=True)

for ax, chain, lbl, start_lbl in zip(
        axes, [chain_x, chain_y],
        ["Cadena X", "Cadena Y"], ["V", "C"]):
    for t in range(len(chain)):
        alpha = 0.4 if (coupling_time and t > coupling_time + 1) else 0.9
        ax.barh(0, 1, left=t, color=VC_COLORS[chain[t]],
                alpha=alpha, edgecolor="white", linewidth=0.3)
    ax.set_yticks([])
    ax.set_ylabel(f"{lbl}\n(inicio={start_lbl})", fontsize=11, fontweight="bold")
    if coupling_time is not None:
        ax.axvline(coupling_time, color=COLORS["dark"], linestyle="--",
                   linewidth=2, zorder=5)

axes[0].set_title(
    "Acoplamiento: dos cadenas, MISMOS números aleatorios, distintos inicios",
    fontsize=13, fontweight="bold")
axes[1].set_xlabel("Paso", fontsize=11)

if coupling_time is not None:
    axes[0].text(coupling_time + 0.5, 0.4,
                 f"\u03c4 = {coupling_time}\n(acopladas)",
                 fontsize=11, color=COLORS["dark"], fontweight="bold",
                 va="center")

legend_elements = [mpatches.Patch(facecolor=VC_COLORS[0], label="V"),
                   mpatches.Patch(facecolor=VC_COLORS[1], label="C")]
axes[0].legend(handles=legend_elements, loc="upper right", fontsize=10)
fig.tight_layout(pad=2)
plt.show()

In [ ]:
n_trials = 500
coupling_times = []

for trial in range(n_trials):
    rng_trial = np.random.default_rng(trial)
    us = rng_trial.random(1000)
    cx, cy = 0, 1
    for t in range(1000):
        u = us[t]
        nx = 0 if u < P_VC[cx, 0] else 1
        ny = 0 if u < P_VC[cy, 0] else 1
        cx, cy = nx, ny
        if cx == cy:
            coupling_times.append(t + 1)
            break
    else:
        coupling_times.append(1000)   # no acopladas en 1000 pasos (raro)

coupling_times = np.array(coupling_times)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(coupling_times, bins=range(1, int(coupling_times.max()) + 2),
        color=COLORS["blue"], edgecolor="white", alpha=0.8)
ax.set_xlabel("Tiempo de acoplamiento \u03c4", fontsize=12)
ax.set_ylabel("Frecuencia", fontsize=12)
ax.set_title("Distribución del tiempo de acoplamiento (500 repeticiones)",
             fontsize=13, fontweight="bold")
ax.axvline(coupling_times.mean(), color=COLORS["red"], linestyle="--",
           linewidth=2, label=f"Media = {coupling_times.mean():.1f}")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"  Media   \u03c4 = {coupling_times.mean():.2f}")
print(f"  Mediana \u03c4 = {np.median(coupling_times):.0f}")
print(f"  Máximo  \u03c4 = {coupling_times.max()}")
print()
print("El acoplamiento SIEMPRE ocurre \u2014 la única pregunta es cuándo.")

## Parte 3 — Tiempo de mezcla y velocidad de convergencia

¿Qué tan rápido converge la cadena?  Depende de la estructura de $P$.

In [ ]:
P_fast = np.array([[0.4, 0.6],
                   [0.5, 0.5]])    # balanceada -> mezcla rápida

P_slow = np.array([[0.99, 0.01],
                   [0.01, 0.99]])  # casi desconectada -> mezcla lenta

def pi_from_P(P):
    vals, vecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(vals - 1.0))
    v = np.real(vecs[:, idx])
    return v / v.sum()

pi_fast = pi_from_P(P_fast)
pi_slow = pi_from_P(P_slow)

n_powers = 200
tv_fast = []
tv_slow = []

Pn_fast = np.eye(2)
Pn_slow = np.eye(2)

for n in range(1, n_powers + 1):
    Pn_fast = Pn_fast @ P_fast
    Pn_slow = Pn_slow @ P_slow
    tv_fast.append(0.5 * np.sum(np.abs(Pn_fast[0] - pi_fast)))
    tv_slow.append(0.5 * np.sum(np.abs(Pn_slow[0] - pi_slow)))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, n_powers + 1), tv_fast, color=COLORS["blue"], linewidth=2,
        label="Mezcla rápida (P_fast)")
ax.plot(range(1, n_powers + 1), tv_slow, color=COLORS["red"], linewidth=2,
        label="Mezcla lenta (P_slow)")
ax.set_xlabel("n  (exponente de $P^n$)", fontsize=12)
ax.set_ylabel("Distancia de variación total a \u03c0", fontsize=12)
ax.set_title("Velocidad de convergencia: $\\|P^n_{0,:} - \\pi\\|_{TV}$",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.set_yscale("log")
ax.set_ylim(1e-15, 1)
plt.tight_layout()
plt.show()

In [ ]:
for name, P in [("P_fast", P_fast), ("P_slow", P_slow)]:
    eigenvalues = np.sort(np.abs(np.linalg.eigvals(P)))[::-1]
    lambda2 = eigenvalues[1]
    print(f"{name}:")
    print(f"  Eigenvalores (|.|): {eigenvalues}")
    print(f"  Segundo mayor eigenvalor |lambda_2| = {lambda2:.4f}")
    print(f"  -> Mientras más cerca de 1, más LENTO converge.")
    print()

print("El segundo eigenvalor controla la velocidad de convergencia.")
print("P_fast tiene |lambda_2| pequeño -> converge rápido.")
print("P_slow tiene |lambda_2| aprox 1 -> converge muy lento.")

## Parte 4 — Rompiendo la ergodicidad

No todas las cadenas convergen.  Veamos qué rompe la ergodicidad.

In [ ]:
P_periodic = np.array([[0.0, 1.0],
                       [1.0, 0.0]])

n_steps = 50
chain = np.zeros(n_steps + 1, dtype=int)
chain[0] = 0
for t in range(n_steps):
    chain[t + 1] = 1 - chain[t]   # determinista: 0->1->0->1...

# Indicador de estar en estado 0 en cada paso
indicator_0 = (chain == 0).astype(float)

fig, ax = plt.subplots(figsize=(12, 4))
ax.step(range(n_steps + 1), indicator_0, where="mid",
        color=COLORS["red"], linewidth=2, label="P(estado = 0 en paso t)")
ax.axhline(0.5, color=COLORS["dark"], linestyle="--", linewidth=1.5,
           label="pi = 0.5 (si convergiera)")
ax.set_xlabel("Paso t", fontsize=12)
ax.set_ylabel("Estado = 0", fontsize=12)
ax.set_title("Cadena periódica: oscila para siempre, no converge",
             fontsize=13, fontweight="bold")
ax.set_ylim(-0.1, 1.1)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
P_reducible = np.array([
    [0.6, 0.4, 0.0, 0.0],
    [0.3, 0.7, 0.0, 0.0],
    [0.0, 0.0, 0.5, 0.5],
    [0.0, 0.0, 0.4, 0.6],
])

n_steps = 500
rng1 = np.random.default_rng(10)
rng2 = np.random.default_rng(20)

cadena_red = CadenaDeMarkov(P_reducible)
chain1 = cadena_red.simular(0, n_steps, rng=rng1)   # componente {0,1}
chain2 = cadena_red.simular(2, n_steps, rng=rng2)   # componente {2,3}

freq1_s0 = np.cumsum(chain1 == 0) / np.arange(1, len(chain1) + 1)
freq2_s0 = np.cumsum(chain2 == 0) / np.arange(1, len(chain2) + 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freq1_s0, color=COLORS["blue"], linewidth=2,
        label="Inicio en estado 0 (componente {0,1})")
ax.plot(freq2_s0, color=COLORS["red"], linewidth=2,
        label="Inicio en estado 2 (componente {2,3})")
ax.set_xlabel("Paso", fontsize=12)
ax.set_ylabel("Fracción de tiempo en estado 0", fontsize=12)
ax.set_title("Cadena reducible: distintos inicios -> distinto comportamiento a largo plazo",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 0.7)
plt.tight_layout()
plt.show()

# Valor teórico para componente {0,1}
pi_comp = CadenaDeMarkov(P_reducible[:2, :2]).estacionaria()
print(f"pi dentro de la componente {{0,1}}: pi_0 = {pi_comp[0]:.3f},  pi_1 = {pi_comp[1]:.3f}")
print(f"Cadena desde estado 2 NUNCA visita estado 0 -> frecuencia = 0")
print()
print("Distintos inicios -> distinto comportamiento a largo plazo -> NO hay pi única.")

In [ ]:
P_fixed = np.array([[0.1, 0.9],
                    [0.8, 0.2]])

n_steps = 200
rng = np.random.default_rng(42)
cadena_fixed = CadenaDeMarkov(P_fixed)
chain_fixed = cadena_fixed.simular(0, n_steps, rng=rng)

pi_fixed = cadena_fixed.estacionaria()
freq_0 = np.cumsum(chain_fixed == 0) / np.arange(1, len(chain_fixed) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: la periódica rota
ax = axes[0]
chain_per = np.zeros(51, dtype=int)
for t in range(50):
    chain_per[t+1] = 1 - chain_per[t]
freq_per = np.cumsum(chain_per == 0) / np.arange(1, 52)
ax.plot(freq_per, color=COLORS["red"], linewidth=2, label="Periódica [[0,1],[1,0]]")
ax.axhline(0.5, color=COLORS["dark"], linestyle="--", linewidth=1.5)
ax.set_title("Periódica: oscila", fontsize=12, fontweight="bold")
ax.set_xlabel("Paso")
ax.set_ylabel("Fracción en estado 0")
ax.set_ylim(0.3, 0.7)
ax.legend(fontsize=9)

# Derecha: la arreglada
ax = axes[1]
ax.plot(freq_0, color=COLORS["blue"], linewidth=2,
        label="Arreglada [[0.1,0.9],[0.8,0.2]]")
ax.axhline(pi_fixed[0], color=COLORS["dark"], linestyle="--", linewidth=2,
           label=f"$\\pi_0 = {pi_fixed[0]:.3f}$")
ax.set_title("Con self-loop: converge", fontsize=12, fontweight="bold")
ax.set_xlabel("Paso")
ax.set_ylabel("Fracción en estado 0")
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

fig.suptitle("Arreglar la periodicidad: agregar self-loops rompe la oscilación",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Parte 5 — LLN para cadenas de Markov: la vindicación de Markov

Nekrasov dijo que la LLN requiere independencia.  Markov mostró que no.

Veamos: ¿converge el promedio temporal tanto para muestras independientes como
para una cadena de Markov?

In [ ]:
n_steps = 3000
target = pi_vc[0]   # pi_V aprox 0.444

# --- Experimento 1: muestras i.i.d. ---
rng_iid = np.random.default_rng(99)
samples_iid = (rng_iid.random(n_steps) < target).astype(float)
running_avg_iid = np.cumsum(samples_iid) / np.arange(1, n_steps + 1)

# --- Experimento 2: cadena de Markov ---
rng_mc = np.random.default_rng(99)
chain_mc = cadena_vc.simular(1, n_steps - 1, rng=rng_mc)  # start at C
f_chain = (chain_mc == 0).astype(float)   # f(V)=1, f(C)=0
running_avg_mc = np.cumsum(f_chain) / np.arange(1, n_steps + 1)

# --- Gráfica ---
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(running_avg_iid, color=COLORS["blue"], linewidth=1.2, alpha=0.8,
        label="Muestras i.i.d.")
ax.plot(running_avg_mc, color=COLORS["red"], linewidth=1.2, alpha=0.8,
        label="Cadena de Markov")
ax.axhline(target, color=COLORS["dark"], linestyle="--", linewidth=2.5,
           label=f"$\\pi_V = {target:.3f}$", zorder=10)

ax.set_xlabel("Número de muestras / pasos", fontsize=12)
ax.set_ylabel("Promedio acumulado de $f(X_t)$", fontsize=12)
ax.set_title("Ambos convergen. La cadena es más lenta pero CONVERGE. Markov tenía razón.",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=12, loc="upper right")
ax.set_ylim(0.2, 0.7)
plt.tight_layout()
plt.show()

print(f"Promedio final i.i.d.:          {running_avg_iid[-1]:.4f}")
print(f"Promedio final cadena Markov:   {running_avg_mc[-1]:.4f}")
print(f"Valor teórico pi_V:             {target:.4f}")

## Resumen y puntos clave

| Concepto | Idea central |
|---|---|
| **Teorema ergódico** | La fracción de tiempo en cada estado converge a $\pi$, sin importar el inicio |
| **Acoplamiento** | Dos cadenas con los mismos números aleatorios se fusionan en tiempo finito |
| **Tiempo de mezcla** | Controlado por $|\lambda_2|$: más cerca de 1 $\to$ más lento |
| **Periodicidad** | Rompe la convergencia puntual (la cadena oscila) |
| **Reducibilidad** | Rompe la unicidad de $\pi$ (distintos inicios $\to$ distintos límites) |
| **LLN de Markov** | El promedio temporal converge incluso **sin independencia** |

**Condiciones para ergodicidad:**
1. **Irreducibilidad** — se puede ir de cualquier estado a cualquier otro
2. **Aperiodicidad** — no hay ciclos deterministas
3. (Para espacios finitos, esto basta)

> *"La cadena olvida su pasado... pero nunca de golpe."*